# Welcome to Modal notebooks!

Write Python code and collaborate in real time. Your code runs in Modal's
**serverless cloud**, and anyone in the same workspace can join.

This notebook comes with some common Python libraries installed. Run
cells with `Shift+Enter`.

In [1]:
!ls /mnt/experiments

README.md  nebius-gpu-hw1  nebius-gpu-hw2  requirements.txt


In [2]:
! cd /mnt/experiments && mkdir nebius-gpu-hw1 && mv *.py nebius-gpu-hw1

In [3]:
%uv pip install -r /mnt/experiments/requirements.txt

Using Python 3.12.6 environment at: /usr/local
Resolved 62 packages in 435ms
⠙ Preparing packages... (0/27)
⠙ Preparing packages... (0/27)
⠙ Preparing packages... (0/27)
pluggy     ------------------------------     0 B/20.06 KiB
⠙ Preparing packages... (0/27)
pluggy     ------------------------------ 14.81 KiB/20.06 KiB
⠙ Preparing packages... (0/27)
aiohappyeyeballs ------------------------------     0 B/14.71 KiB
pluggy     ------------------------------ 14.81 KiB/20.06 KiB
⠙ Preparing packages... (0/27)
aiohappyeyeballs ------------------------------ 14.71 KiB/14.71 KiB
pluggy     ------------------------------ 14.81 KiB/20.06 KiB
⠙ Preparing packages... (0/27)
iniconfig  ------------------------------     0 B/7.31 KiB
aiohappyeyeballs ------------------------------ 14.71 KiB/14.71 KiB
pluggy     ------------------------------ 14.81 KiB/20.06 KiB
⠙ Preparing packages... (0/27)
iniconfig  ------------------------------ 7.31 KiB/7.31 KiB
aiohappyeyeballs -----------------------------

In [1]:
import os
os.chdir("/mnt/experiments")

In [2]:
%uv run python nebius-gpu-hw1/hw1_task.py

HW1: GPU Roofline Model — NVIDIA L40S 48GB GDDR6

Theoretical specs (FP32):
  Peak compute:    92 TFLOP/s
  Peak bandwidth:  0.86 TB/s
  Ridge point:     106.0 FLOP/Byte

Running benchmarks (first run compiles kernels via torch.compile)...
  lowest-AI:  0.829 ms | BW: 0.65 TB/s
    1 ops (   eager): 2.072 ms | AI: 0.0833 FLOP/B | 0.06 TFLOP/s
    1 ops (compiled): 0.809 ms | AI: 0.25 FLOP/B | 0.17 TFLOP/s
    2 ops (   eager): 4.622 ms | AI: 0.0833 FLOP/B | 0.06 TFLOP/s
    2 ops (compiled): 0.810 ms | AI: 0.5 FLOP/B | 0.33 TFLOP/s
    4 ops (   eager): 9.739 ms | AI: 0.0833 FLOP/B | 0.06 TFLOP/s
    4 ops (compiled): 0.809 ms | AI: 1 FLOP/B | 0.66 TFLOP/s
    8 ops (   eager): 19.987 ms | AI: 0.0833 FLOP/B | 0.05 TFLOP/s
    8 ops (compiled): 0.809 ms | AI: 2 FLOP/B | 1.33 TFLOP/s
   16 ops (   eager): 40.467 ms | AI: 0.0833 FLOP/B | 0.05 TFLOP/s
   16 ops (compiled): 0.809 ms | AI: 4 FLOP/B | 2.65 TFLOP/s
   32 ops (   eager): 81.459 ms | AI: 0.0833 FLOP/B | 0.05 TFLOP/s
   32 ops (c

In [4]:
!mkdir nebius-gpu-hw2 && mv *.py nebius-gpu-hw2

In [5]:
!cat `/mnt/experiments/nebius-gpu-hw2/hw2_task.py`

/usr/bin/sh: 1: /mnt/experiments/nebius-gpu-hw2/hw2_task.py: Permission denied



In [3]:
import os
os.chdir("/mnt/experiments")

In [2]:
%uv run python nebius-gpu-hw2/hw2_task.py

Note: you may need to restart the kernel to use updated packages.


In [4]:
"""Utilities for HW2 — provided, do not modify."""

import time
from pathlib import Path

import torch
from transformers import LlamaConfig, LlamaForCausalLM

SEED = 0
MODEL_NAME = "Tiny random Llama (2 layers, d_model=2048)"
VOCAB_SIZE = 4096
PROMPT_LEN = 1024
MAX_NEW_TOKENS = 128
PROFILE_STEPS = 12
RESULTS_DIR = Path("nebius-gpu-hw2") / "results"
RESULTS_DIR.mkdir(exist_ok=True)


def build_model(dtype):
    """Create a tiny decoder-only model with real attention and KV cache."""
    torch.manual_seed(SEED)
    config = LlamaConfig(
        vocab_size=VOCAB_SIZE,
        hidden_size=2048,
        intermediate_size=6144,
        num_hidden_layers=2,
        num_attention_heads=8,
        num_key_value_heads=8,
        max_position_embeddings=PROMPT_LEN + MAX_NEW_TOKENS + 64,
        bos_token_id=1,
        eos_token_id=2,
        pad_token_id=0,
        tie_word_embeddings=False,
    )
    model = LlamaForCausalLM(config)
    model.to(device="cuda", dtype=dtype)
    model.eval()
    return model


def get_input_ids():
    generator = torch.Generator(device="cuda")
    generator.manual_seed(SEED)
    return torch.randint(
        low=0,
        high=VOCAB_SIZE,
        size=(1, PROMPT_LEN),
        generator=generator,
        device="cuda",
        dtype=torch.long,
    )


def slow_loop(model, input_ids, n_steps):
    """Reference slow generation loop — do not modify."""
    generated_ids = input_ids.clone()
    generated_tokens = []
    for _ in range(n_steps):
        outputs = model(input_ids=generated_ids)
        next_token_id = torch.argmax(outputs.logits[:, -1, :], dim=-1)
        token_value = next_token_id.item()
        generated_tokens.append(token_value)
        generated_ids = torch.cat([generated_ids, next_token_id.unsqueeze(0)], dim=1)
    return generated_tokens


def time_generation(loop_fn, model, input_ids, label):
    """Time loop_fn for MAX_NEW_TOKENS with proper GPU synchronization."""
    torch.cuda.synchronize()
    start = time.perf_counter()
    generated_tokens = loop_fn(model, input_ids, MAX_NEW_TOKENS)
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    preview = generated_tokens[:8]
    print(
        f"{label}: {MAX_NEW_TOKENS} tokens in {elapsed:.2f}s "
        f"({MAX_NEW_TOKENS / elapsed:.1f} tok/s)"
    )
    print(f"Token preview: {preview}")
    return elapsed


In [12]:
import torch



def optimized_loop(model, input_ids, n_steps):
    # TODO: fix the performance issues you found — changes may include
    # both `optimized_loop` and `generate_optimized`
    # preallocate tehnsor?
    generated_ids = input_ids.clone()
    generated_tokens = []
    for _ in range(n_steps):
        outputs = model(input_ids=generated_ids)
        next_token_id = torch.argmax(outputs.logits[:, -1, :], dim=-1)
        token_value = next_token_id.item() # this op is wasteful as pytorchneeds to sybnchornise
        generated_tokens.append(token_value)
        generated_ids = torch.cat([generated_ids, next_token_id.unsqueeze(0)], dim=1)
    return generated_tokens


def profile(loop_fn, model, input_ids, trace_name: str):
    # TODO: wrap loop_fn(model, input_ids, PROFILE_STEPS) with torch.profiler,
    # print the summary table, and export a Chrome trace to RESULTS_DIR / trace_name
    with torch.profiler.profile(
            activities=[
                torch.profiler.ProfilerActivity.CPU,
                torch.profiler.ProfilerActivity.CUDA,
            ]
    ) as p:
        loop_fn(model, input_ids, PROFILE_STEPS)
    print(p.key_averages().table(sort_by="self_cuda_time_total", row_limit=-1))
    p.export_chrome_trace(str(RESULTS_DIR / trace_name))


def generate_optimized(optimized_trace_name: str) -> float:
    # TODO: load the model (consider dtype and other loading options),
    # then call profile() and time_generation() on optimized_loop.
    # Return the elapsed time from time_generation so main() can print a speedup.
    model = build_model(torch.bfloat16)
    input_ids = get_input_ids()
    profile(optimized_loop, model, input_ids, optimized_trace_name)
    return time_generation(optimized_loop, model, input_ids, "optimised")







In [14]:
def main(optimized_trace_name="v1_optimized_trace.json"):
    print("=" * 60)
    print("HW2: LLM Inference Optimization")
    print(f"Model: {MODEL_NAME}")
    print("=" * 60)

    print("\n--- Part 1: Slow baseline ---")
    model = build_model(torch.float32)
    input_ids = get_input_ids()
    profile(slow_loop, model, input_ids, "v0_slow_trace.json")
    slow_elapsed = time_generation(slow_loop, model, input_ids, "Slow")
    del model
    torch.cuda.empty_cache()

    print("\n--- Part 2: Optimized ---")
    optimized_elapsed = generate_optimized(optimized_trace_name=optimized_trace_name)

    print("\n" + "=" * 60)
    print("SUMMARY")
    print("=" * 60)
    if optimized_elapsed is None or optimized_elapsed <= 0:
        print("generate_optimized() did not return a positive elapsed time; "
              "cannot compute speedup.")
    else:
        speedup = slow_elapsed / optimized_elapsed
        print(f"  Slow:      {slow_elapsed:6.2f}s")
        print(f"  Optimized: {optimized_elapsed:6.2f}s")
        print(f"  Speedup:   {speedup:6.2f}x  (vs V0 slow baseline)")

In [15]:
main("only_bfloat16")

HW2: LLM Inference Optimization
Model: Tiny random Llama (2 layers, d_model=2048)

--- Part 1: Slow baseline ---
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                               aten::mm         2.36%       2.861ms        12.27%      14.849ms      82.496us      91.569ms        86.37%      91.569ms     508.714us           180  
                                 ampere_sgemm_128x64_tn         0.00%       0.000us         0.

In [ ]:
def optimized_loop(model, input_ids, n_steps):
    batch, _ = input_ids.shape
    new_tokens = torch.empty(
        (batch, n_steps), dtype=input_ids.dtype, device=input_ids.device
    )
    generated_ids = input_ids
    for i in range(n_steps):
        outputs = model(input_ids=generated_ids)
        next_token_id = torch.argmax(outputs.logits[:, -1, :], dim=-1)
        new_tokens[:, i] = next_token_id
        generated_ids = torch.cat([generated_ids, next_token_id.unsqueeze(1)], dim=1)
    return new_tokens[0].tolist()

main("bf16_and_preallocation_no_items")

HW2: LLM Inference Optimization
Model: Tiny random Llama (2 layers, d_model=2048)

--- Part 1: Slow baseline ---
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                               aten::mm         2.47%       2.964ms         6.34%       7.597ms      42.208us      86.309ms        86.26%      86.309ms     479.494us           180  
                                 ampere_sgemm_128x64_tn         0.00%       0.000us         0.

In [18]:
def optimized_loop(model, input_ids, n_steps):
    batch, _ = input_ids.shape
    new_tokens = torch.empty(
        (batch, n_steps), dtype=input_ids.dtype, device=input_ids.device
    )

    past_key_values = None
    cur = input_ids
    for i in range(n_steps):
        outputs = model(input_ids=cur, past_key_values=past_key_values, use_cache=True)
        past_key_values = outputs.past_key_values
        next_token_id = torch.argmax(outputs.logits[:, -1, :], dim=-1)
        new_tokens[:, i] = next_token_id
        cur = next_token_id.unsqueeze(1)

    return new_tokens[0].tolist()

main("bf16_preallocation_no_items_kvcaching")

HW2: LLM Inference Optimization
Model: Tiny random Llama (2 layers, d_model=2048)

--- Part 1: Slow baseline ---
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                               aten::mm         2.25%       2.654ms         5.16%       6.078ms      33.768us      90.754ms        86.38%      90.754ms     504.190us           180  
                                 ampere_sgemm_128x64_tn         0.00%       0.000us         0.

In [11]:
main(optimized_trace_name="v2_optimized_trace.json")

HW2: LLM Inference Optimization
Model: Tiny random Llama (2 layers, d_model=2048)

--- Part 1: Slow baseline ---
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                               aten::mm         2.40%       2.836ms         4.91%       5.808ms      32.269us      98.658ms        86.45%      98.658ms     548.102us           180  
                                 ampere_sgemm_128x64_tn         0.00%       0.000us         0.